In [2]:
%load_ext sagemaker_studio_analytics_extension.magics
%sm_analytics emr connect --verify-certificate False --cluster-id j-21LB0UJKB0IQY --auth-type Basic_Access --language python  --emr-execution-role-arn arn:aws:iam::396913739385:role/data-scientist

Successfully read emr cluster(j-21LB0UJKB0IQY) details
Initiating EMR connection..
WARN: Skipping SSL certificate verification because verify_certificate option is set to "False". We recommended that you enable SSL certificate verification. Please run the command %sm_analytics? for details about enabling SSL certificate verification.
Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1777080821389_0001,pyspark,idle,Link,Link,MKDVX6ASA2GVIMUBYCQWX457ODTDH6GG,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.
{"namespace": "sagemaker-analytics", "cluster_id": "j-21LB0UJKB0IQY", "error_message": null, "success": true, "service": "emr", "operation": "connect"}


In [8]:
import boto3

sm = boto3.client("sagemaker")

response = sm.list_training_jobs(MaxResults=1)  # solo para probar conexión
print("Conexión OK")

Conexión OK


In [10]:
from sagemaker.serverless import ServerlessInferenceConfig

In [11]:
import sagemaker
from sagemaker.model import Model
from sagemaker import image_uris
from sagemaker.serverless import ServerlessInferenceConfig

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()

print("Region:", region)
print("Role:", role)

Region: us-east-1
Role: arn:aws:iam::396913739385:role/BBVARole-ada-us-east-1-sbx-live-m-size-dev


In [12]:
# tu tar con SOLO inference.py
model_s3_path = "s3://ada-us-east-1-sbx-live-co-risk-data/O017332/Model/inference.tar.gz"

# (referencia) carpeta donde están los artefactos
# debe coincidir con lo que usas dentro de inference.py
artifacts_prefix = "O017332/Model/"
bucket = "ada-us-east-1-sbx-live-co-risk-data"

In [13]:
!aws s3 cp $model_s3_path inference.tar.gz
!tar -tzf inference.tar.gz

download: s3://ada-us-east-1-sbx-live-co-risk-data/O017332/Model/inference.tar.gz to ./inference.tar.gz
._inference.py
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
inference.py


In [16]:
from sagemaker import image_uris

image_uri = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.0",
    py_version="py310",
    image_scope="inference",  
    instance_type="ml.m5.large"
)

print("Image URI:", image_uri)

Image URI: 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:2.0-cpu-py310


In [17]:
from sagemaker.model import Model

model = Model(
    model_data="s3://ada-us-east-1-sbx-live-co-risk-data/O017332/inference.tar.gz",
    role=role,
    entry_point="inference.py",
    image_uri=image_uri
)

print("Modelo creado correctamente")

Modelo creado correctamente


In [18]:
from sagemaker.serverless import ServerlessInferenceConfig

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,   
    max_concurrency=1
)

print("Serverless config OK")

Serverless config OK
